In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent,Runner,trace,function_tool,OpenAIChatCompletionsModel,input_guardrail,GuardrailFunctionOutput
from typing import Dict
from groq import Groq
import sendgrid
import os
from sendgrid.helpers.mail import Mail,Email,To,Content,From
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [3]:
groq_Api_key = os.getenv("GROQ_API_KEY")
deepseek_api_key =os.getenv("DEEPSEEK_API_KEY")


if groq_Api_key:
    print(f"groq api key exists and begin with:{groq_Api_key[:8]}")

else:
    print("groq api doent exist")


if  deepseek_api_key:
    print(f"deepseek_api_key exists and begin with:{deepseek_api_key[:5]}")

else:
    print("deepseek api key doesnt exist")


groq api key exists and begin with:gsk_vtVz
deepseek_api_key exists and begin with:sk-15


In [4]:
instruction1 = "you are a sale agent working for abcAI.\
    a company provide check user satisfcation for sales items SOC 2 compliance and SOC 2 audits. you worte professional,serious cold emails"


instruction2="you are a hunorous,engaging sales agent working for abcAI.\
      a company provide check user satisfcation for sales items SOC 2 compliance and SOC 2 audits. you worte FUNNY, cold emails ARE LIKELY TO GT A RESPONSE "

instruction3="you are a busy sales agent working in abcAI.\
    A COMPANY PROVIDE CHECK USER SATISFICATION FOR SALES ITEMS SOC 2 compliance and SOC 2 audits.,write concise,to the point cold emails "

In [5]:
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL ="https://api.groq.com/openai/vi"

In [6]:
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL ,api_key=groq_Api_key)
deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL,api_key=deepseek_api_key)


groq_model =OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile",openai_client=groq_client)   #like  (client.chat.completions.create)
deepseek_model =OpenAIChatCompletionsModel(model="deepseek chat" ,openai_client=deepseek_client)

In [7]:
##create agent
sale_agent1 = Agent(name="deepseek sales agent" ,instructions=instruction1,model=deepseek_model)
sale_agent2 = Agent(name="groq sales agent",instructions=instruction2,model=groq_model)

In [8]:
description ="write a cold sale email"

tool1 =sale_agent1.as_tool(tool_name="sales_agent1",tool_description=description)
tool2 =sale_agent2.as_tool(tool_name="sale_agent2",tool_description=description)

In [9]:
@function_tool
def send_html_email(subject:str,html_body:str) ->Dict[str,str]:
    """send an email with the give subject and html body to all customers"""
    sg =sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    From_email = Email("sanojcsam123@gmail.com")
    to_email = To("sanojcsam26@gmail.com")
    mail = Mail(from_email,to_email,subject,content).get()
    response= sg.client.mail.send.post(request_body=mail)
    return {"status":"success"}


In [10]:
subject_instructions = "you can write a subject for cold sales email. \
    you are given a message and need to write a subject for an email."

html_instructions= "you can convert a text email body to an html email body. \
    you are given a text email body which might have some markdown and convert it to html email body"

In [11]:

subject_writer =Agent(name="email subject writer" ,instructions=subject_instructions,model=groq_model)
subject_tool =subject_writer.as_tool(tool_name="subject writer" ,tool_description="write a subject for cold sales email")

email_html_convertor =Agent(name="email html body convertor",instructions=html_instructions,model=groq_model)
html_tool =email_html_convertor.as_tool(tool_name="html body convertor",tool_description="convert text email body to html email body")


In [12]:
email_tools = [subject_tool ,html_tool, send_html_email]

In [13]:
instructions= "you are an email formatter and sender.you recevive the body of an email to be sent.\
    first use the subject_writer to write a subject for the email,then use the html_converter tool to convert the body to html.\
        finally,send_html_email tool use send the email with the subject and html body"

In [14]:
emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools =email_tools,
    model=groq_model,
    handoff_description= "convert an email to html and send it"
)

In [15]:
tools=[tool1,tool2]
handoffs=[emailer_agent]

In [16]:
sales_manager_instructions= "you are a sales manager working in abcAI company you use the tools and generate sale emails. \
    you try all 2 sales agent tools after select best one from these.\
    you select best one after picking the email,you handoff to the Email Manager agent to format and send the email"    

In [17]:
sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=groq_model
)
message ="send out a cold sales email addressed to dear hr ceo"


with trace("automated SDR"):
    result = await Runner.run(sales_manager,message)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


NotFoundError: Error code: 404 - {'error': {'message': 'Unknown request URL: POST /openai/vi/chat/completions. Please check the URL for typos, or see the docs at https://console.groq.com/docs/', 'type': 'invalid_request_error', 'code': 'unknown_url'}}

In [19]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name:str


guardrail_agent = Agent(
    name="Name check",
    instructions="check if the user is including someones personal name in what they want you to do",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

In [20]:
@input_guardrail
async def guardrail_name_check(content,agent,message):
    result = await Runner.run(guardrail_agent,message,content=content.content)
    is_name_in_message =result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name":result.final_output},tripwire_triggered=is_name_in_message)


In [21]:
sales_manager_instructions= "you are a sales manager working in abcAI company you use the tools and generate sale emails. \
    you try all 2 sales agent tools after select best one from these.\
    you select best one after picking the email,you handoff to the Email Manager agent to format and send the email"   

In [24]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions =sales_manager_instructions,
    tools =tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_name_check]

)


message="Send out a cold sale email addressed Dear CEO from bob"

with trace("protected automated sdr"):
    result = Runner.run(careful_sales_manager,message)

OPENAI_API_KEY is not set, skipping trace export
